# Personal Job Agent

You've got a job description you need to apply for and email? Use me!

In [ ]:
from typing import Annotated, TypedDict, List, Dict, Any, Optional
from dotenv import load_dotenv
import nest_asyncio

import requests
import os
import gradio as gr
import sqlite3

from IPython.display import Image, display

from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
# from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.tools import tool

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser
from langchain_community.utilities import GoogleSerperAPIWrapper

from db_handler import load_to_store, read_pdf_file
from schema import ResumeData

import base64

import sendgrid
from sendgrid.helpers.mail import Email, Mail, Content, To, Attachment

In [ ]:
load_dotenv(override=True)

SENDGRID_API_KEY = os.environ.get("SENDGRID_API_KEY")


GPT_OSS_MODEL = "gpt-oss:20b" # Or whatever model you have pulled
LLAMA_MODEL = "llama3.2:latest" # Or whatever model you have pulled

DB = "resume.db" # The directory where the ChromaDB will be stored

EMBEDDINGS = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

VECTORSTORE = Chroma(
    persist_directory=DB,
    embedding_function=EMBEDDINGS
)

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    resume_id: str | None
    db_status: str | None
    resume_path: str | None
    email_sent: bool
    resume_data: Optional[ResumeData]

config = {"configurable": {"thread_id": "1"}}

graph_builder = StateGraph(State)

db_path = "memory.db"
conn = sqlite3.connect(db_path, check_same_thread=False)
sql_memory = SqliteSaver(conn)

nest_asyncio.apply()

## Tools

In [ ]:
serper = GoogleSerperAPIWrapper()

@tool
def tool_search(query: str) -> str:
    """Useful for when you need more information from an online search"""
    return serper.run(query)

In [ ]:
from langsmith import uuid


@tool
def populate_db_tool(resume_path: str) -> dict:
    """Populate the database with resume data"""
    resume = read_pdf_file(resume_path)
    if resume is None:
        return {"status": "error", "message": "Invalid resume file"}
    resume_id = str(uuid.uuid4())

    chunk_count = load_to_store(resume, VECTORSTORE, resume_id)

    return {
        "status": "success",
        "resume_id": resume_id,
        "chunks_added": chunk_count,
        "resume_path": resume_path
    }

In [ ]:
@tool
def send_email_tool(email_data: ResumeData, job_role: str) -> str:
    """Send an email with the given subject, body and a resume attachment"""
    sg = sendgrid.SendGridAPIClient(api_key=SENDGRID_API_KEY)
    from_email = Email("hello@oluseun.dev")
    to_email = To(email_data.cover_letter_data.hiring_manager_email)
    content = Content("text/html", email_data.cover_letter_data.cover_letter)
    attachment = Attachment(
        file_content=base64.b64encode(open(email_data.resume_path, "rb").read()).decode(),
        file_type="application/pdf",
        file_name=os.path.basename(email_data.resume_path),
    )
    mail = Mail(from_email, to_email, job_role, content, attachments=[attachment]).get()
    response = sg.client.mail.send.post(request_body=mail)
    print("Email response", response.status_code)
    return "success"

In [ ]:
async_browser =  create_async_playwright_browser(headless=False)  # headful mode
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
playwright_tools = toolkit.get_tools()

In [ ]:
tools = [tool_search, send_email_tool] + playwright_tools

In [ ]:
gpt_llm = ChatOllama(
    model=GPT_OSS_MODEL,
    temperature=0,
)

db_worker_llm_with_tools = gpt_llm.bind_tools([populate_db_tool])

gpt_oss_llm = ChatOllama(
    model=GPT_OSS_MODEL,
    temperature=0,
)

send_email_llm_with_tools = gpt_oss_llm.bind_tools([send_email_tool])

llama = ChatOllama(
    model=LLAMA_MODEL,
    temperature=0,
)

cover_letter_llm = llama.with_structured_output(ResumeData)

## Nodes

In [ ]:
def ingest_resume(state: State):
    resume_path = state.get("resume_path")

    if not resume_path:
        return state

    result = populate_db_tool.invoke({"resume_path": resume_path})

    return {
        "resume_id": result["resume_id"],
        "db_status": result["status"],
        "resume_path": result["resume_path"],
    }

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# User message looks like: "what job roles can I apply for with this resume?"

def roles_parser(state: State):
    query = state["messages"][-1].content
    resume_id = state["resume_id"]

    docs = VECTORSTORE.similarity_search(
        query,
        filter={"resume_id": resume_id}
    )

    system = f"Suggest job roles based on this resume:\n{docs}"

    response = gpt_llm.invoke([
        SystemMessage(content=system),
        HumanMessage(content=query)
    ])

    return {"messages": state["messages"] + [response]}

In [ ]:
from gradio.external import format_conversation
from langchain_core.messages import HumanMessage

# user expected to chose from the list of job roles and ask the assistant to create a cover letter for a specific role. The message might look like: "create a cover letter for the software engineer role at google"

def email_creator(state: State):
    query = state["messages"][-1].content
    resume_id = state["resume_id"]

    docs = VECTORSTORE.similarity_search(
        query,
        filter={"resume_id": resume_id}
    )

    system = f"""
Generate a cover letter using this resume context:
{docs}

Include resume_path: {state.get("resume_path")}
"""

    result = cover_letter_llm.invoke([
        SystemMessage(content=system),
        HumanMessage(content=query)
    ])

    return {
        "resume_data": result,
        "messages": state["messages"] + [
            SystemMessage(content="Cover letter generated. Awaiting approval.")
        ]
    }

In [ ]:
def send_email(state: State) -> Dict[str, Any]:
    messages = state["messages"]
    last_message = messages[-1]

    system_message = f"""You are a helpful assistant that sends an email to the hiring manager with the candidate's cover letter and resume attached. 
    Use the send_email_tool to send the email. Here is the cover letter data {state.get("resume_data").cover_letter_data} and the resume path {state.get("resume_data").resume_path}.

    Once done, return a message confirming that the email has been sent successfully.
    """

    found_system_message = False
    messages = state["messages"]
    for message in messages:
        if isinstance(message, SystemMessage):
            message.content = system_message
            found_system_message = True
    
    if not found_system_message:
        messages = [SystemMessage(content=system_message)] + messages

    response = send_email_llm_with_tools.invoke([SystemMessage(content=system_message), HumanMessage(content=last_message.content)])

    return {
        "messages": [response],
    } 

## Bringing it all together

In [ ]:
# Add nodes
from langgraph.graph import END
    
def human_approval(state: State):
    return interrupt({
        "cover_letter": state["resume_data"].cover_letter_data.cover_letter
    })
    
def approval_router(state: State):
    if not state.get("approved"):
        return "END"
    if not SENDGRID_API_KEY:
        return "END"
    return "send_email"


graph_builder.add_node("ingest_resume", ingest_resume)
graph_builder.add_node("roles_parser", roles_parser)
graph_builder.add_node("email_creator", email_creator)
graph_builder.add_node("human_approval", human_approval)
graph_builder.add_node("send_email", send_email)

# Flow
graph_builder.add_edge(START, "ingest_resume")
graph_builder.add_edge("ingest_resume", "roles_parser")
graph_builder.add_edge("roles_parser", "email_creator")
graph_builder.add_edge("email_creator", "human_approval")

graph_builder.add_conditional_edges(
    "human_approval",
    approval_router,
    {
        "send_email": "send_email",
        "END": END,
    },
)

In [ ]:
graph = graph_builder.compile(checkpointer=sql_memory)

graph.get_state(config)

display(Image(graph.get_graph().draw_mermaid_png()))